In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIO\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIOAfter\BoSSSpad.dll"
#r "D:\Users\miao\Documents\JupyterNotebook\BoSSSpadMiao\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Text;
using System.Globalization;
using System.Threading;
using System.Threading.Tasks;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
//ExecutionQueues

In [3]:
//GetDefaultQueue()

In [4]:
static var myBatch = ExecutionQueues[0];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("EE-FS-Taylor-Couette-Flow");

In [5]:
BoSSSshell.WorkflowMgm.Init("EE-FS-Taylor-Couette-Flow");

Project name is set to 'EE-FS-Taylor-Couette-Flow'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\EE-FS-Taylor-Couette-Flow'.


In [6]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

Opening existing database 'D:\Users\miao\Documents\Database\EE-FS-Taylor-Couette-Flow'.


## Create grid

In [18]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid(int h) { 
        double xLeft = -2;
        double xRight = 2;
        double yTop = 2;
        double yBottom = -2;
        int kelem = Convert.ToInt32(Math.Pow(2, h));
        
        double[] CutOut1Point1 = new double[2] {  0.5, -0.5 }; 
        double[] CutOut1Point2 = new double[2] {  -0.5, 0.5 };
        
        var CutOut1 = new BoSSS.Platform.Utils.Geom.BoundingBox(2);
        CutOut1.AddPoint(CutOut1Point1);
        CutOut1.AddPoint(CutOut1Point2);

        double[] Xnodes = GenericBlas.Linspace(xLeft, xRight, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, CutOuts: CutOut1);

        grd.EdgeTagNames.Add(1, "wall_lower");
        grd.EdgeTagNames.Add(2, "wall_upper");
        grd.EdgeTagNames.Add(3, "wall_left");
        grd.EdgeTagNames.Add(4, "wall_right");
        grd.EdgeTagNames.Add(5, "wall_inside");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 5;

            if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                et = 2;
            if(Math.Abs(X[0] - xLeft) <= 1.0e-8)
                et = 3;
            if(Math.Abs(X[0] - xRight) <= 1.0e-8)
                et = 4;

            return et;
        });

        return grd;
     }
 
 }

In [20]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double InletVelocityX(double[] X) {");
           stw.WriteLine("    double A = 0.922554663967561;");
           stw.WriteLine("    double B = -1.690218655870246;");
           stw.WriteLine("    double v_x = -(A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[1] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return v_x;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double InletVelocityY(double[] X) {");
           stw.WriteLine("    double A = 0.922554663967561;");
           stw.WriteLine("    double B = -1.690218655870246;");
           stw.WriteLine("    double v_y = (A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[0] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return v_y;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double PreDisplacementX(double[] X, double t) {");
           stw.WriteLine("    double A = 0.067608746234810;");
           stw.WriteLine("    double B = -0.033804373117405;");
           stw.WriteLine("    double dis_x = -(A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[1] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return dis_x;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double PreDisplacementY(double[] X, double t) {");
           stw.WriteLine("    double A = 0.067608746234810;");
           stw.WriteLine("    double B = -0.033804373117405;");
           stw.WriteLine("    double dis_y = (A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[0] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return dis_y;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double Pressure(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return -1;");
           stw.WriteLine("  }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return -(X[1] * X[1] + X[0] * X[0] - (1 + 0.25 * Math.Sqrt(2)) * (1 + 0.25 * Math.Sqrt(2)));");
           stw.WriteLine("  }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InletVelocityX(){
        return new Formula("BoundaryValues.InletVelocityX", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InletVelocityY(){
        return new Formula("BoundaryValues.InletVelocityY", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_PreDisplacementX(){
        return new Formula("BoundaryValues.PreDisplacementX", true, AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_PreDisplacementY(){
        return new Formula("BoundaryValues.PreDisplacementY", true, AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_Pressure(){
        return new Formula("BoundaryValues.Pressure", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [23]:
public static ZLS_Control Couette( int p = 2, int h = 2, int AMRlvl = 2, double BeamDensity = 1) {
    ZLS_Control C = new ZLS_Control(p);
    C.AgglomerationThreshold = 0.2;
    C.NoOfMultigridLevels = 1;
    //int D = 2;
    AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

    #region db
    C.savetodb = true;
    C.ProjectName = "Couette";
    C.SessionName = "Fluid-Solid-Taylor-Couette-p" + p + "-h" + h;
    C.ContinueOnIoError = false;
    //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
    //C.PostprocessingModules.Add(new MovingContactLineLogging());
    #endregion
    
    // Physical Parameters
    // ===================
    #region physics
    C.PhysicalParameters.rho_A = 1.3;
    //C.PhysicalParameters.rho_B = 2.0;
    C.PhysicalParameters.mu_A = 0.2;
    //C.PhysicalParameters.mu_B = 0.1;
    double sigma = 0.9;
    C.PhysicalParameters.Sigma = sigma;
    //C.PhysicalParameters.betaS_A = 0.0;
    //C.PhysicalParameters.betaS_B = 0.0;
    //C.PhysicalParameters.betaL = 0.0;
    //C.PhysicalParameters.theta_e = Math.PI / 2.0;
    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = false; //??
    C.Material = new Solid() {
        Density = BeamDensity,
        //Lame2 = 1000,
        //Viscosity = 0.1
        Lame2 = 1e1,
        Viscosity = 0.2
    };
    #endregion
    // grid generation
    // ===============
    #region grid
    C.SetGrid(GridFactory.GenerateGrid(h));
    #endregion
    // Initial Values
    // ==============
    //
    C.AddInitialValue("VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddInitialValue("VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    C.AddInitialValue("DisplacementX#C", BoundaryValueFactory.Get_PreDisplacementX());
    C.AddInitialValue("DisplacementY#C", BoundaryValueFactory.Get_PreDisplacementY());
    C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_Zero());
    C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
    C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
    //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());
    // boundary conditions
    // ===================
    #region BC
    C.AddBoundaryValue("wall_lower", "VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_lower", "VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_upper", "VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_upper", "VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_left", "VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_left", "VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_right", "VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_right", "VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_inside", "DisplacementX#A", BoundaryValueFactory.Get_PreDisplacementX());
    C.AddBoundaryValue("wall_inside", "DisplacementY#A", BoundaryValueFactory.Get_PreDisplacementY());



    C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
    C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
    C.PhysicalParameters.sliplength = 0.0;
    #endregion
    // misc. solver options
    // ====================
    #region solver
    //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
    //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
    //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;
    C.NonLinearSolver.MaxSolverIterations = 80;
    C.NonLinearSolver.MinSolverIterations = 2;
    //C.Solver_MaxIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    //C.Solver_ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-12;
    //C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;
    //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
    //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
    //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
    //C.fullReInit = false; 
    C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
    C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.Standard;
    C.AdaptiveMeshRefinement = false;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband { maxRefinementLevel = AMRlvl });
    C.AMR_startUpSweeps = AMRlvl;
    #endregion

    C.DynamicLoadBalancing_On = false;
    //C.DynamicLoadBalancing_Period = 500;
    C.DynamicLoadBalancing_RedistributeAtStartup = false;

    //C.GridPartType = GridPartType.clusterHilbert;
    C.GridPartType = GridPartType.METIS;

    // Timestepping
    // ============
    #region time
    //C.CheckJumpConditions = true;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Newton;
    
    C.TimesteppingMode = compMode;
    double dt = 1;
    C.dtMax = dt;
    C.dtMin = dt;
    C.NoOfTimesteps = 20;
    C.saveperiod = 1;
    #endregion
    return C;
}


(57,7): error CS1061: 'ZLS_Control' does not contain a definition for 'ExactSolution_provided' and no accessible extension method 'ExactSolution_provided' accepting a first argument of type 'ZLS_Control' could be found (are you missing a using directive or an assembly reference?)



Error: compilation error

## Send and run jobs

In [36]:
double[] Density = new double[] {0.1}; 
int[] Resolution = new int[] {3, 4, 5, 6}; 
int[] DgOrder = new int[] {1, 2, 3, 4}; 

In [38]:
foreach(int DgO in DgOrder){
foreach(int Res in Resolution){
foreach(double Den in Density){
    var C_CTRLFILE = Couette(DgO, Res, 0, Den);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = 1;
    JobLocal.NumberOfThreads = 1;
    JobLocal.Activate(myBatch);
    JobLocal.ShowOutput(); 
}
}
}



Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Fluid-Solid-Taylor-Couette-p1-h3 ... 
Opening existing database 'D:\Users\miao\Documents\Database\EE-FS-Taylor-Couette-Flow'.
Set Database: { Session Count = 16; Grid Count = 34; Path = \\dc3\userspace\miao\cluster\EE-FS-Taylor-Couette-Flow }
Grid is not in database yet...
Grid successfully saved: 64727271-2884-4da2-902a-d1ad70ed3f7a


Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\miao\cluster\Deployment\EE-FS-Taylor-Couette-Flow-ZwoLevelSetSolver2024Aug26_163255.095614
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Starting external console ...
(You may close the new window at any time, the job will continue.)


Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


...

Deploying job Fluid-Solid-Taylor-Couette-p4-h5 ... 
Set Database: { Session Count = 27; Grid Count = 48; Path = \\dc3\userspace\miao\cluster\EE-FS-Taylor-Couette-Flow }
Grid is not in database yet...
Grid successfully saved: 64e92ce0-0e21-4c37-aa51-a85c8467b825


Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\miao\cluster\Deployment\EE-FS-Taylor-Couette-Flow-ZwoLevelSetSolver2024Aug26_163727.596231
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Starting external console ...
(You may close the new window at any time, the job will continue.)


Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Fluid-Solid-Taylor-Couette-p4-h6 ... 
Set Database: { Session Count = 29; Grid Count = 49; Path = \\dc3\userspace\miao\cluster\EE-FS-Taylor-Couette-Flow }
Grid is not in database yet...
Grid successfully saved: 3a12f091-559a-48d5-8bd9-1afe721e3674


Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\miao\cluster\Deployment\EE-FS-Taylor-Couette-Flow-ZwoLevelSetSolver2024Aug26_163754.884127
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Starting external console ...
(You may close the new window at any time, the job will continue.)


In [63]:
wmg.Sessions

#0: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p4-h6	08/26/2024 15:18:24	12eee9c1...
#1: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p3-h6	08/26/2024 15:18:06	f0747343...
#2: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p2-h6	08/26/2024 15:16:17	7529dbbe...
#3: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p3-h5	08/26/2024 15:17:47	43e01ce8...
#4: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p4-h5	08/26/2024 15:17:43	e3e94693...
#5: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p4-h3	08/26/2024 15:18:26	34f1835a...
#6: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p3-h4	08/26/2024 15:17:13	292922fb...
#7: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p4-h4	08/26/2024 15:18:39	056a61e0...
#8: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p1-h6	08/26/2024 15:15:17	df71f7a8...
#9: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p2-h5	08/26/2024 15:15:47	cd9e9af5...
#10: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-

In [ ]:
//BoSSSshell.WorkflowMgm.Sessions.ForEach(si => si.Delete(true));

In [30]:
wmg.Sessions[0].Timesteps.Count

21

In [46]:
var outPath = wmg.Sessions[4].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: d:\Users\miao\AppData\Local\BoSSS\plots\sessions\EE-FS-Taylor-Couette-Flow__Fluid-Solid-Taylor-Couette-p3-h6__f0747343-c108-44a0-9ed2-e8ccd3764d49


## Post processing

In [35]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [37]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

In [37]:
wmg.Sessions

#0: EE-FS-Taylor-Couette-Flow	1Fluid-Solid-Taylor-Couette-p3-h4	08/26/2024 14:49:11	92790f2a...
#1: EE-FS-Taylor-Couette-Flow	1Fluid-Solid-Taylor-Couette-p2-h3	08/26/2024 14:55:41	453ecf31...
#2: EE-FS-Taylor-Couette-Flow	Fluid-Solid-Taylor-Couette-p3-h4	08/26/2024 14:39:05	7d9761d1...


In [29]:
var session = wmg.Sessions[0];

In [31]:
session.Timesteps.Count

21

In [33]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("DisplacementX").ProbeAt(0.0, -1.3),
        t => "DisplacementX"
        );

In [34]:
mydataset.PlotNow()

Using gnuplot: d:\Users\miao\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.06188 
 
 
 
 
 0.06189 
 
 
 
 
 0.0619 
 
 
 
 
 0.06191 
 
 
 
 
 0.06192 
 
 
 
 
 0.06193 
 
 
 
 
 0.06194 
 
 
 
 
 0.06195 
 
 
 
 
 0.06196 
 
 
 
 
 0.06197 
 
 
 
 
 0.06198 
 
 
 
 
 0 
 
 
 
 
 5 
 
 
 
 
 10 
 
 
 
 
 15 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DisplacementX 
 
 
 DisplacementX

In [ ]:
for (int i = 0; i < 5; i++) 
{
  var DispX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();
  var VeloX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "VelocityX").Single();
  Console.WriteLine(DispX.ProbeAt(0.0, 0.25));
  //Console.WriteLine(VeloX.ProbeAt(0.0, 1.0));
}

In [ ]:
int[] Cases = new int[] {0, 1, 3, 4}; 

In [ ]:
foreach(int i in Cases){
  var DispX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();
  var VeloX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "VelocityX").Single();
  Console.WriteLine(DispX.ProbeAt(0.0, 0.25));
  Console.WriteLine(VeloX.ProbeAt(0.0, 1.0));
}

In [50]:
var DispX = session.Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();

In [51]:
DispX.ProbeAt(0.0, 0.25)

0.0008327781162742156

In [53]:
Displacement


(1,1): error CS0103: The name 'Displacement' does not exist in the current context



Error: compilation error

## Restart

In [ ]:
databases[0].Sessions

In [ ]:
//var Sloc = databases[0].Sessions.Last();
var Sloc = databases[0].Sessions[0];
Sloc

In [ ]:
var c2 = Sloc.CreateRestartControl() as ZLS_Control;

In [ ]:
c2.GetType()

In [ ]:
c2.GridGuid

In [ ]:
Sloc.Timesteps.Last().GridID

In [ ]:
c2.GridGuid = Sloc.Timesteps.Last().GridID;

In [ ]:
c2.GridGuid

In [ ]:
//c2.DynamicLoadBalancing_On = true;
//c2.DynamicLoadBalancing_Period = 10;
c2.DynamicLoadBalancing_RedistributeAtStartup = false;
//c2.GridPartType = GridPartType.METIS;

//c2.AdaptiveMeshRefinement = true;
//c2.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 5});
//c2.AMR_startUpSweeps = 5;
c2.AdaptiveMeshRefinement = false;

In [ ]:
//c2.TimeSteppingScheme = TimeSteppingScheme.BDF2;

In [ ]:
var JobLocal2 = c2.CreateJob();
JobLocal2.Status

In [ ]:
JobLocal2.NumberOfMPIProcs = 2;
JobLocal2.Activate();
JobLocal2.ShowOutput();

In [ ]:
databases[0].Sessions

In [ ]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

In [ ]:
databases[0].Sessions[0].